# What To Expect In This Notebook
This notebook serves as documentation of the process that I went through to engineer the features that will soon become useful for the classifier that I am building.

It includes:
- Initial feature ideas and assumptions
- Feature revisions and design iterations
- Analysis of what worked and what didn't
- Reflections on feature effectiveness

## Problem Definition

Build a scheduling system that adapts to a student's workload and risk of burnout.

## Key Challenge

Students experience academic workload differently. Features must capture both personal well-being and the demands of a student's schedule while being sensitive to different types of students.

## Goal

Create features that contribute to predicting trends in student burnout before it happens.


# Initial Thoughts

In our project, a daily rating system was implemented to guage the importance of seven features that contribute most directly to burnout on a scale of 0 (no effect) - 10 (great effect):

Daily Rating Features:
- `stress`
- `energy` (a person can have a lot of energy despite having little rest)
- `mood` 
- `sleep`
- `time_working` (0-24)
- `burnout` (our truth)

Additionally, we need to consider the student's schedule in terms of tasks such as assignments, labs, exmas nad other deadlines. Student's on the app can assign different tasks with set priorities (0-5). Exams however, are different in practice as events such as midterms or finals should  By checking the "Exam" box, the priority is instantly set to a special priority tier (6). This is to exaggerate the added stress of exams during exam periods. We will call this number assignment

- `priority`

## Initial Features

At first glance, averaging our Daily Rating Features is an immediete and easy feature to implement. By averaging our Daily Ratings across day intervals (ex 3,7,semester history) we can get the most immediate trend of a student's patience. So our features would look like:

- `avg_stress3`
- `avg_stress7`
- `avg_energy3`
- `avg_energy7`
- etc.

To assess immediate catalysts for burnout counting the number of tasks due within the next few days is a natural choice. We do have to account for the fact that tasks spread out evenly over a week are different. Having 5 tasks within 3 days is different than 5 spread out over a week, and this difference is even more exaggerated when it comes to exams.

- `tasks_in3`
- `tasks_in7`
- `exams_in3`
- `exams_in7`
- `days_until_next_exam`


To assess the threat of multiple deadlines, we track the sum of priorities within a time period to see how the student reacts using `priority_sum7`. Likewise, an indicator for handling large amounts of workloads should also be assessed. Using the feature `load` (tasks due / average weekly tasks) we can normalize a student's schedule by comparing the amount of tasks they have to the amount of classes they have. For example, if there is one task for each class within a week `load` = `1.00`. Although there can be multiple tasks for one course (labs, assignments, quizzes) we can still investigate a range that is "normal" for the student.

## Hypothesis

- `stress`, `energy`, `mood`, and `sleep` contribute the most to burnout (at varying degrees depending on person to person)
- Logging tasks due within `n` days follows closely. They are likely correlated to `stress`, `energy`, `mood`, and `sleep`.
- `days_until_next_exam` is not necessarily a useful feature. All other features are direct measurements of workload in some capacity, however depending on the student, `days_until_next_exam` can matter. In a sense we can expect vary degrees of correlation between `burnout` and `days_until_next_exam` on a student to student basis.

# Evaluation and Refinement #1 - Initial Stage

This section explains the changes to our initial model and why. Many of these changes were made during development. That being said, expect the following to be closer to my actual process than documentation of what changes were made and why so expect to see some features to be removed and reintroduced later.

### Replacing Averages With Weighted Averages

Averages are a good way to capture the most common or most typical value of a dataset, however data can be flawed. Semesters come in highs and lows, the beginning of the semester is going to be easier than the middle of the semester, and the midterm season is going to be more hectic than what is considered normal. Our dataset is small since it is a semester to semester basis, and therefore is going to be very sensitive to shifts that occur for one or two weeks. Additionally, burnout is measured due to current events so how much of our history should we consider. Should data from a month ago play a big role in today's prediction?

Initially, we measured averages within the 3 days, 7 days, and the entire semester to try and account for this. Very quickly we can get multiple features that may or may not even describe the data well. We can't just have a feature for every 4 day interval in history to account for this and so we are moving to a moving weighted average so that we can penalize old data while adding some extra weight to the most recent changes.

### Removing Load For Alternatives 

`load` requires a lot of extra work in terms of getting the numbers and functions required to calculate it. After trying to implement it I scrapped the idea in favor of implementing features that are more simple to implement that describe the same thing: work to courseload ratio. After careful consideration, I realized that the `priority_sum7` already describes the upcoming workload already. 

### Utilizing Priority 
One issue with using `priority_sum7` is that it fails to consider the effect of larger course loads. To fix this, we can divide `prioirty_sum7` by the amount of tasks to produce an `avgppt` (priority per task). Another way of fixing this is to simply multiply `# of Courses` by `AvgPriority`. I am going to choose the first method since it is easier to normalize, though it is important to note that the ladder is more exaggerative. 

Averages can also obscure outliers. In our case, we want to be sensitive to dramatic changes to immediately. In terms of a confusion matrix we may use for our model later, we would rather have a False Negative (predict burntout while student isn't burntout.) Therefore implementing `max_priority7` accounts for a high priority task that can get drowned out in averages.

### Reflections
- Features should stay simple. Multiple simple features can describe more and are easier to set up than few, complex ones.
- From initial hypothesis, some features such as `days_until_next_exam` are conditionally correlated. It may describe some datasets very well and others not so much.
- Features should be imperfect 



## Feature Development Notes

While developing the first feature functions, several limitations became clear:

- Future work should explore better ways to normalize workload across different semester structures, such as rolling baselines, week-of-semester comparisons, or user-specific historical trends.

